#### What is fine tuning?

Fine tuning is the process of traing a pre-trained large language model and training it futher on the specific dataset so it performs better on specific task and domain.

* Step 1: Pretained model
* Step 2: Prepare Dataset
        You create examples in input → expected output format. Thousands/millions of examples are used.
* Step 3: Train Further

        During training:

            * model makes prediction
            * compare with correct answer
            * calculate error (loss)
            * adjust weights using backpropagation

        Same ML training process.

            Prediction → Compare → Calculate Loss → Update Weights

* Step 4: New specialized model<br>
        After training:
            Base Model + Domain Knowledge = Fine Tuned Model
        
        Example:
            GPT Model + Legal documents = Legal Assistant


#### Before going into deep, let's uderstand what is weight, bias and parametes in LLM and how are they related.

A neural netwoek learns by adjusting numbers internally. Those numbers are
 
    * Weights -> A weight tells the model how important an input is.
    * Biases -> Bias helps shift the output so the model is more flexible.

-> Together they are called parameters.
* parametes -> weight + bias

    Parameters are the numerical values inside the model that get adjusted during the training. Throung these adjustments the model encodes patters and relationships it has learned from the taining data which are then used to make predictions.

    Training Data → Model learns patterns → Patterns get encoded into parameters

Example:

Imagine predicting whether someone will get a job.

    Inputs:

        * Skills
        * Experience
        * Degree

    Not all inputs matter equally.

    * Skills = very important
    * Experience = medium important
    * Degree = less important

    model assigns weights:

    * Skills → 0.8
    * Experience → 0.5
    * Degree → 0.2

    Output = (Input × Weight) + Bias


    Input1 ----(W1=0.8)----
                            \
                             → Sum → + Bias → Output
                            /
    Input2 ----(W2=0.3)----

    Output = (Input1 × W1) + (Input2 × W2) + Bias

During training.

* Initally model starts with random weights

    Eg:
        Weight = 0.1
        Bias = 0.2

* Prdiction is wrong

    Eg:
        Weight = 0.6
        Bias = 0.8

* Prediction improves. This repeats millions of times.

#### Backpropagation update weights

During training:

    Input → Prediction → Compare with actual answer → Calculate error → Update weights → Repeat

Backpropagation is the algorithm that tells the model Which weights caused the error, and by how much should each weight change?

Let's say we have one neuron

    y = (x * w) + b

x = 2, w = 3, b = 1

-> Prediction

y = (2 × 3) + 1
y = 7

-> Actula = 5 (Expected)

* Step 1: Calculate loss (Error)

        Mean Squared Error (MSE)

        Loss = (Predction - Actual) ** 2

        Loss = (7 - 5)²
        Loss = 4

        Model wants loss to be zero

* Step 2: Find Which Weight Caused Error

        * We calculate how much does the loss change if the weight changes?

            ∂Loss / ∂Weight -> gradient (How sensitive is error to this weight?)

        * Large gradient: Weight strongly affecting error
        * Small gradient: Weight has little effect

* Step 3: Chain Rule (Core of Backpropagation)

        Loss depends on Prediction
        Prediction depends on Weight

            We use calculus.

                ∂Loss/∂Weight = ∂Loss/∂Prediction × ∂Prediction/∂Weight

* Step 4: Calcualte gradient:

        We know:

                Loss = (y - actual)²  # y -> prediction
                y = (x × w) + b

        Part 1:

            ∂Loss/∂y = 2(y - actual)

            = 2(7 - 5)
            = 4
        
        Part 2:

            ∂y/∂w = x  #y = (x × w) + b

            = 2

        ∂Loss/∂w = 4 × 2 = 8 

        Gradient = 8

* Step 5: Update Weight

        New Weight = Old Weight - Learning Rate × Gradient

        Assume Learning Rate = 0.1

        New Weight = 3 - (0.1 × 8)
        New Weight = 2.2

* Step 6: Predict Again

        y = (2 × 2.2) + 1
        y = 5.4

        Loss = (5.4 - 5) ** 2
        Loss = 0.16 # Loss reduced model improved.

#### Types of Fine-tuning:

1. Full Fine-Tuning

        7 billion parameters → update all 7 billion

Pros:

* best performance

Cons:

* expensive
* needs large GPU memory 

2. Parameter Efficient Fine Tuning (PEFT)

        Only train small subset.

        Examples:

            LoRA
            QLoRA
            Adapters

Pros:

* cheaper
* faster
* less GPU required

3. Instruction Fine-Tuning

        * Teach model how to follow instructions better.

        * Models like ChatGPT use this heavily.




In [1]:
import torch
import torch.nn as nn

In [2]:
# simple linear model
model=nn.Linear(
    in_features=1,
    out_features=1
)

print(model.weight)
print(model.bias)

Parameter containing:
tensor([[-0.5644]], requires_grad=True)
Parameter containing:
tensor([0.2686], requires_grad=True)


In [3]:
# Data set

X = torch.tensor(
    [
        [20.0],
        [25.0],
        [30.0]
    ]
)

Y = torch.tensor(
    [
        [40.0],
        [50.0],
        [60.0]
    ]
)

In [4]:
# Loss Function

loss_fn = nn.MSELoss()

In [5]:
# Optimizer - updates parameters.
# We use SGD (gradient descent).

optimizer = torch.optim.SGD(
    params=model.parameters(),
    lr=0.0001
)

In [6]:
# Trainig loop

for epoch in range(100):

    # forward pass
    prediction = model(X) 

    # calualte loss
    loss = loss_fn(prediction, Y) # Loss = (y-x)**2

    # Clear old gradients
    optimizer.zero_grad()

    # backpropagation
    loss.backward()

    #update parameters
    optimizer.step()  # weight = weight - learning_rate × gradient

    if epoch % 10 == 0:
        print(
            "Epoch:",
            epoch,
            "Loss:",
            loss.item()
        )


Epoch: 0 Loss: 4185.322265625
Epoch: 10 Loss: 267.1786193847656
Epoch: 20 Loss: 17.0589542388916
Epoch: 30 Loss: 1.0922826528549194
Epoch: 40 Loss: 0.0730191245675087
Epoch: 50 Loss: 0.007951905950903893
Epoch: 60 Loss: 0.003798188641667366
Epoch: 70 Loss: 0.0035325668286532164
Epoch: 80 Loss: 0.0035151976626366377
Epoch: 90 Loss: 0.0035138584207743406


In [7]:
print("Weight:", model.weight)
print("Bias:", model.bias)

Weight: Parameter containing:
tensor([[1.9857]], requires_grad=True)
Bias: Parameter containing:
tensor([0.3678], requires_grad=True)


In [8]:
# test

test = torch.tensor(
    [20.0]
)

res = model(test)
res

tensor([40.0811], grad_fn=<ViewBackward0>)

#### Fine-tuning a spam classification model

In [9]:
corpuses = [
    "win free money now",
    "meeting at 5 pm",
    "claim your prize",
    "project discussion tomorrow"
]

labels = [
    1,   # spam
    0,   # not spam
    1,
    0
]

In [10]:
import re
def clean_corpus(corpus:str) -> str:
    return re.sub('[^a-zA-z]', ' ', corpus)

In [11]:
cleand_corpuses = list(map(clean_corpus, corpuses))

In [12]:
#Bag of Words
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer()
#cv=CountVectorizer(binary=True)

In [13]:
X=cv.fit_transform(cleand_corpuses)
cv.vocabulary_

{'win': 11,
 'free': 3,
 'money': 5,
 'now': 6,
 'meeting': 4,
 'at': 0,
 'pm': 7,
 'claim': 1,
 'your': 12,
 'prize': 8,
 'project': 9,
 'discussion': 2,
 'tomorrow': 10}

In [14]:
X = torch.tensor(
    X.toarray(),
    dtype=torch.float32
)

Y = torch.tensor(
    labels,
    dtype=torch.float32
).reshape(-1, 1)

In [15]:
import torch.nn as nn


class SpamClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.linear = nn.Linear(
            in_features=len(cv.vocabulary_),      # input size
            out_features=1
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        x = self.linear(x)

        x = self.sigmoid(x)

        return x


model = SpamClassifier()

In [16]:
loss_fn = nn.BCELoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

In [17]:
for epoch in range(500):

    prediction = model(X)

    loss = loss_fn(
        prediction,
        Y
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch % 50 == 0:
        print(
            "Epoch:",
            epoch,
            "Loss:",
            loss.item()
        )

Epoch: 0 Loss: 0.6749228835105896
Epoch: 50 Loss: 0.5860679149627686
Epoch: 100 Loss: 0.5134882926940918
Epoch: 150 Loss: 0.4539032578468323
Epoch: 200 Loss: 0.4046342968940735
Epoch: 250 Loss: 0.3635563850402832
Epoch: 300 Loss: 0.32900863885879517
Epoch: 350 Loss: 0.2997000217437744
Epoch: 400 Loss: 0.2746267020702362
Epoch: 450 Loss: 0.25300517678260803


In [18]:
test_corpus = "You are rich"
test=cv.transform([test_corpus])
print(test.toarray())

test = torch.tensor(
    test.toarray(),
    dtype=torch.float32
)
test.shape

[[0 0 0 0 0 0 0 0 0 0 0 0 0]]


torch.Size([1, 13])

In [19]:
res = model(test)
res

tensor([[0.5261]], grad_fn=<SigmoidBackward0>)

In [20]:
# Fine tune

new_sentences = [
    "You are rich now",
    "team meeting tomorrow",
    "You got gold"
]

new_labels = [
    1,
    0,
    1
]

In [21]:
cleand_new_corpuses = list(map(clean_corpus, new_sentences))

In [22]:
X_new=cv.transform(cleand_new_corpuses)

In [23]:
X_new = torch.tensor(
    X_new.toarray(),
    dtype=torch.float32
)

Y_new = torch.tensor(
    new_labels,
    dtype=torch.float32
).reshape(-1, 1)


In [24]:
X_new.shape

torch.Size([3, 13])

In [25]:
for epoch in range(500):

    prediction = model(X_new)

    loss = loss_fn(
        prediction,
        Y_new
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch % 50 == 0:
        print(
            "Epoch:",
            epoch,
            "Loss:",
            loss.item()
        )

Epoch: 0 Loss: 0.49437808990478516
Epoch: 50 Loss: 0.4636707305908203
Epoch: 100 Loss: 0.4364694654941559
Epoch: 150 Loss: 0.41219112277030945
Epoch: 200 Loss: 0.3903771936893463
Epoch: 250 Loss: 0.3706634044647217
Epoch: 300 Loss: 0.3527563810348511
Epoch: 350 Loss: 0.3364170789718628
Epoch: 400 Loss: 0.32144835591316223
Epoch: 450 Loss: 0.30768582224845886


In [26]:
test_corpus_new = "You are rich"
test_new=cv.transform([test_corpus_new])
print(test_new.toarray())

test_new = torch.tensor(
    test_new.toarray(),
    dtype=torch.float32
)
test_new.shape

[[0 0 0 0 0 0 0 0 0 0 0 0 0]]


torch.Size([1, 13])

In [27]:
res = model(test)
res

tensor([[0.6636]], grad_fn=<SigmoidBackward0>)